# Grid Mainline Training Notebook

This notebook keeps only the shortest GridEnv training path.

- Build the default grid configuration with `compose_experiment_config()`
- Construct a training runner and execute training
- Inspect reward decomposition after training
- Save the final checkpoint

Run all cells in order to complete one normal grid-aware training run.
            


In [ ]:
from pathlib import Path
import sys
import warnings

try:
    get_ipython().run_line_magic('load_ext', 'autoreload')
    get_ipython().run_line_magic('autoreload', '2')
except Exception:
    pass

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / 'configs').exists():
    project_root = project_root.parent
if not (project_root / 'configs').exists():
    raise RuntimeError('Could not locate the project root.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

warnings.filterwarnings('ignore', message='The behavior of DataFrame concatenation with empty or all-NA entries is deprecated.*')
project_root
            


In [ ]:
import torch

from configs import compose_experiment_config
from scripts.plots.reward_plots import plot_reward_decomposition
from scripts.utils.experiment_notebook_utils import build_runner, get_madrl_checkpoint_root, summarize_cfg
from scripts.utils.torch_runtime import configure_torch_runtime, describe_device
            


In [ ]:
algorithm = 'MADDPG'
train_episodes = 256
reward_plot_window = 10
seed = 7
runtime_mode = 'performance'
device_request = 'cuda' if torch.cuda.is_available() else 'cpu'
require_cuda = False

cfg = compose_experiment_config(
    profile='debug',
    algorithm=algorithm,
    model_family='mlp',
    vec_env_type='subproc',
    data_dir=project_root / 'data',
    device=device_request,
    runtime_mode=runtime_mode,
    seed=seed,
    require_cuda=require_cuda,
)

cfg.train.train_episodes = train_episodes
cfg.train.num_envs = 16
cfg.train.vec_env_type = 'subproc'
cfg.train.batch_size = 8192
cfg.train.buffer_size = 100000
cfg.train.update_interval = 8
cfg.train.updates_per_step = 8
cfg.train.use_noise_decay = True
cfg.train.noise_std_init = 0.35
cfg.train.noise_std_min = 0.05
cfg.train.max_train_steps = None
cfg.train.noise_decay_steps = cfg.train.train_episodes * cfg.env.episode_limit

runtime_state = configure_torch_runtime(cfg, device=device_request, seed=seed, require_cuda=require_cuda)
summary = summarize_cfg(cfg)
summary['device_info'] = describe_device(runtime_state)
summary
            


In [ ]:
runner = build_runner(cfg, seed=seed, env_name='GridTrainMainline', number=1)
episodes_completed = runner.run()
print(f'Training finished: {episodes_completed} episodes')
runner.perf_summary
            


In [ ]:
plot_reward_decomposition(
    history=list(runner.history),
    episode_rewards=runner.episode_rewards,
    reward_fn=runner.env_evaluate.reward_fn,
    title='Training Reward Decomposition',
    window=reward_plot_window,
)
            


In [ ]:
save_dir = get_madrl_checkpoint_root(project_root) / f'{algorithm}_Grid_Mainline'
save_dir.mkdir(parents=True, exist_ok=True)
runner.save_model(str(save_dir), episode=episodes_completed)
runner.close()
print(f'Model saved to: {save_dir}')
            
